# 02 — Lagrangian ICNC + LWC Evolution Along Plume Trajectories

Replicate the Omanovic et al. (2025, JAMES) f05/f12 analysis style:
ICNC and relative LWC change time series along Lagrangian plume tracks.
Overlays COSMO-SPECS spectral-bin output with HOLIMO observations.
Highlights differences in WBF efficiency between spectral-bin and bulk approaches.

In [5]:
from pathlib import Path
import sys
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

src_dir = next(p / "src" for p in (Path.cwd(), *Path.cwd().parents)
               if (p / "src" / "polarcap_runtime.py").is_file())
sys.path.insert(0, str(src_dir))

from utilities import load_plume_path_runs
from utilities.holimo_helpers import load_and_prepare_holimo
from utilities.model_helpers import calculate_mean_diameter

In [6]:
# --- Configuration ---
PROCESSED_ROOT = Path("../../data/processed")
HOLIMO_FILE = "../../data/observations/holimo_data/CL_20230125_1000_1140_SM058_SM060_ts1.nc"
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

SEEDING_START = np.datetime64("2023-01-25T10:28:00")
TIME_WINDOW_MIN = (0, 15)  # minutes after seeding

# HOLIMO mission plume time windows (from holimo_helpers.py)
HOLIMO_PLUME_WINDOWS = {
    'SM058': ('2023-01-25T10:36:00', '2023-01-25T10:41:00'),
    'SM059': ('2023-01-25T10:57:00', '2023-01-25T11:03:00'),
    'SM060': ('2023-01-25T11:25:00', '2023-01-25T11:29:00'),
}
HOLIMO_BGD_WINDOW = ('2023-01-25T10:20:00', '2023-01-25T10:30:00')

# Ice concentration threshold for plume definition (cm⁻³, Omanovic 2025 style)
ICNC_THRESHOLD = 0.001  # L⁻¹ (using L⁻¹ because model output is converted to L⁻¹)


In [7]:
# --- Load model data ---
datasets = load_plume_path_runs(processed_root=PROCESSED_ROOT, kinds=("integrated",))
run_label = list(datasets.keys())[0]
ds = datasets[run_label]["integrated"]
print(f"Run: {run_label}, dims: {dict(ds.dims)}")

Run: cs-eriswil__20251125_114053:20251125114238, dims: {'cell': 3, 'time': 248, 'diameter': 66, 'diameter_edges': 67}


/var/folders/g1/3_czjq2s0ms47mpj5clrshbr0000gp/T/ipykernel_90323/336817953.py:5: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"Run: {run_label}, dims: {dict(ds.dims)}")


In [8]:
# --- Compute elapsed time and bulk quantities per cell ---
elapsed_s = (ds.time - SEEDING_START).values / np.timedelta64(1, 's')
elapsed_min = elapsed_s / 60.0

# ICNC: sum nf over diameter bins (already in L⁻¹)
icnc_per_cell = ds['nf'].sum(dim='diameter')  # (time, cell)

# CDNC: sum nw over diameter bins (1/cm³)
if 'nw' in ds.data_vars:
    cdnc_per_cell = ds['nw'].sum(dim='diameter')  # (time, cell)
else:
    cdnc_per_cell = None
    print("Warning: 'nw' not in dataset; CDNC comparison not available")

# LWC: sum qw over bins (g/m³)
if 'qw' in ds.data_vars:
    lwc_per_cell = ds['qw'].sum(dim='diameter')  # (time, cell)
elif 'qwtot' in ds.data_vars:
    lwc_per_cell = ds['qwtot']
else:
    lwc_per_cell = None
    print("Warning: 'qw' not in dataset; LWC comparison not available")

# IWC: sum qf or qi over bins (g/m³)
if 'qf' in ds.data_vars:
    iwc_per_cell = ds['qf'].sum(dim='diameter')
elif 'qi' in ds.data_vars:
    iwc_per_cell = ds['qi'].sum(dim='diameter') if 'diameter' in ds['qi'].dims else ds['qi']
else:
    iwc_per_cell = None

# Mean ice diameter
diameters = ds.diameter.values
nf_vals = ds['nf'].values
mean_ice_diam = np.full((ds.sizes['time'], ds.sizes['cell']), np.nan)
for c in range(ds.sizes['cell']):
    nf_ma = np.ma.masked_less_equal(nf_vals[:, :, c], 0)
    try:
        # The shape of nf_vals[:,:,c] is (time, diameter), so we might need to transpose if calculate_mean_diameter expects diameter on last axis.
        # calculate_mean_diameter operates on the last axis, which is diameter here. 
        # However, checking the error, it might be safer to manually do the weighted sum:
        mask = nf_vals[:, :, c] > 0
        num = np.sum(nf_vals[:, :, c] * diameters * mask, axis=1)
        den = np.sum(nf_vals[:, :, c] * mask, axis=1)
        mean_ice_diam[:, c] = np.where(den > 0, num / den, np.nan)
    except Exception as e:
        print(f"Error calculating mean diam: {e}")

print(f"ICNC range: {float(icnc_per_cell.min()):.4f} – {float(icnc_per_cell.max()):.2f} L⁻¹")

Error calculating mean diam: operands could not be broadcast together with shapes (3,248) (66,) 
Error calculating mean diam: operands could not be broadcast together with shapes (3,248) (66,) 
Error calculating mean diam: operands could not be broadcast together with shapes (3,248) (66,) 
ICNC range: 0.0000 – 690.27 L⁻¹


In [9]:
# --- Compute ensemble statistics across cells (Omanovic 2025 style) ---
# Filter: only where ICNC > threshold
icnc_arr = icnc_per_cell.values  # (time, cell)
icnc_masked = np.where(icnc_arr > ICNC_THRESHOLD, icnc_arr, np.nan)

icnc_median = np.nanmedian(icnc_masked, axis=1)
icnc_p25 = np.nanpercentile(icnc_masked, 25, axis=1)
icnc_p75 = np.nanpercentile(icnc_masked, 75, axis=1)

# Mean diameter ensemble stats
diam_masked = np.where(icnc_arr > ICNC_THRESHOLD, mean_ice_diam, np.nan)
diam_median = np.nanmedian(diam_masked, axis=1)
diam_p25 = np.nanpercentile(diam_masked, 25, axis=1)
diam_p75 = np.nanpercentile(diam_masked, 75, axis=1)

# CDNC stats
if cdnc_per_cell is not None:
    cdnc_arr = cdnc_per_cell.values
    cdnc_median = np.nanmedian(cdnc_arr, axis=1)
    cdnc_p25 = np.nanpercentile(cdnc_arr, 25, axis=1)
    cdnc_p75 = np.nanpercentile(cdnc_arr, 75, axis=1)

# LWC stats and relative change
if lwc_per_cell is not None:
    lwc_arr = lwc_per_cell.values
    lwc_median = np.nanmedian(lwc_arr, axis=1)
    # Relative LWC change from background (first 2 minutes as background)
    bgd_mask = (elapsed_min >= -5) & (elapsed_min < 0)
    if bgd_mask.sum() > 0:
        lwc_bgd = np.nanmedian(lwc_arr[bgd_mask, :])
    else:
        lwc_bgd = np.nanmedian(lwc_arr[:5, :])
        
    rel_lwc_change = (lwc_median - lwc_bgd) / (lwc_bgd + 1e-10) * 100
    rel_lwc_p25 = (np.nanpercentile(lwc_arr, 25, axis=1) - lwc_bgd) / (lwc_bgd + 1e-10) * 100
    rel_lwc_p75 = (np.nanpercentile(lwc_arr, 75, axis=1) - lwc_bgd) / (lwc_bgd + 1e-10) * 100
else:
    rel_lwc_change = None

print(f"Ensemble cells: {ds.sizes['cell']}")

ValueError: operands could not be broadcast together with shapes (3,248) (248,3) () 

In [ ]:
# --- Load HOLIMO data ---
ds_hol, _, _ = load_and_prepare_holimo(HOLIMO_FILE)

# Extract HOLIMO ice and water concentrations per mission window
hol_missions = {}
for mission, (t0, t1) in HOLIMO_PLUME_WINDOWS.items():
    sel = ds_hol.sel(time=slice(t0, t1))
    ice_c = sel['Ice_concentration'].values
    water_c = sel['Water_concentration'].values
    water_content = sel['Water_content'].values
    hol_missions[mission] = {
        'ice_conc': ice_c[ice_c > 0],
        'water_conc': water_c,
        'water_content': water_content,
    }

# Background HOLIMO LWC
hol_bgd = ds_hol.sel(time=slice(*HOLIMO_BGD_WINDOW))
lwc_bgd_hol = float(hol_bgd['Water_content'].median())

# Relative LWC change for each mission
for mission, data in hol_missions.items():
    data['rel_lwc_change'] = (data['water_content'] - lwc_bgd_hol) / (lwc_bgd_hol + 1e-10) * 100
    print(f"{mission}: median ice conc = {np.nanmedian(data['ice_conc']):.3f} L⁻³, "
          f"median ΔLWC = {np.nanmedian(data['rel_lwc_change']):.1f}%")

In [ ]:
# --- Figure: 2×2 panel (Omanovic 2025 f12 style) ---
fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex='col')
t_mask = (elapsed_min >= TIME_WINDOW_MIN[0]) & (elapsed_min <= TIME_WINDOW_MIN[1])
t_plot = elapsed_min[t_mask]

# --- Top left: ICNC ---
ax = axes[0, 0]
ax.fill_between(t_plot, icnc_p25[t_mask], icnc_p75[t_mask], alpha=0.25, color='C0')
ax.plot(t_plot, icnc_median[t_mask], color='C0', lw=1.5, label='COSMO-SPECS')
# HOLIMO box plots at approximate elapsed times
for i, (mission, data) in enumerate(hol_missions.items()):
    if len(data['ice_conc']) > 0:
        elapsed_approx = (i + 1) * 4 + 4  # rough elapsed minutes mapping to SM058, 59, 60
        ax.boxplot([data['ice_conc']], positions=[elapsed_approx], widths=0.8,
                   patch_artist=True, showfliers=False,
                   boxprops={'facecolor': 'C3', 'alpha': 0.5},
                   medianprops={'color': 'k'})
ax.set(ylabel='ICNC (L$^{-1}$)', title='(a) Ice crystal number concentration')
ax.legend(fontsize=8)

# --- Top right: Mean ice diameter ---
ax = axes[0, 1]
ax.fill_between(t_plot, diam_p25[t_mask], diam_p75[t_mask], alpha=0.25, color='C0')
ax.plot(t_plot, diam_median[t_mask], color='C0', lw=1.5, label='COSMO-SPECS')
ax.set(ylabel='Mean ice diameter (µm)', title='(b) Mean ice crystal diameter')
ax.legend(fontsize=8)

# --- Bottom left: CDNC ---
ax = axes[1, 0]
if cdnc_per_cell is not None:
    ax.fill_between(t_plot, cdnc_p25[t_mask], cdnc_p75[t_mask], alpha=0.25, color='C2')
    ax.plot(t_plot, cdnc_median[t_mask], color='C2', lw=1.5, label='COSMO-SPECS')
# HOLIMO water concentration
for i, (mission, data) in enumerate(hol_missions.items()):
    elapsed_approx = (i + 1) * 4 + 4
    ax.boxplot([data['water_conc'][~np.isnan(data['water_conc'])]], positions=[elapsed_approx],
               widths=0.8, patch_artist=True, showfliers=False,
               boxprops={'facecolor': 'C3', 'alpha': 0.5},
               medianprops={'color': 'k'})
ax.set(xlabel='Elapsed time (min)', ylabel='CDNC (cm$^{-3}$)',
       title='(c) Cloud droplet number concentration')
ax.legend(fontsize=8)

# --- Bottom right: Relative LWC change ---
ax = axes[1, 1]
if rel_lwc_change is not None:
    ax.fill_between(t_plot, rel_lwc_p25[t_mask], rel_lwc_p75[t_mask], alpha=0.25, color='C2')
    ax.plot(t_plot, rel_lwc_change[t_mask], color='C2', lw=1.5, label='COSMO-SPECS ΔLWC')
    ax.axhline(0, color='grey', ls=':', lw=0.8)
# HOLIMO relative LWC change
for i, (mission, data) in enumerate(hol_missions.items()):
    elapsed_approx = (i + 1) * 4 + 4
    valid = data['rel_lwc_change'][~np.isnan(data['rel_lwc_change'])]
    if len(valid) > 0:
        ax.boxplot([valid], positions=[elapsed_approx], widths=0.8,
                   patch_artist=True, showfliers=False,
                   boxprops={'facecolor': 'C3', 'alpha': 0.5},
                   medianprops={'color': 'k'})
ax.set(xlabel='Elapsed time (min)', ylabel='Relative ΔLWC (%)',
       title='(d) Liquid water depletion (WBF signature)')
ax.legend(fontsize=8)

fig.suptitle('Lagrangian plume evolution: COSMO-SPECS vs HOLIMO\n'
             '(following Omanovic et al. 2025 analysis framework)',
             fontsize=10, y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'figure_lagrangian_icnc_lwc.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'figure_lagrangian_icnc_lwc.png'}")

In [ ]:
# --- Summary statistics table ---
print("\n=== Lagrangian Plume Statistics ===")
print(f"Time window: {TIME_WINDOW_MIN[0]}–{TIME_WINDOW_MIN[1]} min after seeding")
print(f"Ensemble cells: {ds.sizes['cell']}")
print(f"\nModel (COSMO-SPECS):")
print(f"  Peak ICNC:          {np.nanmax(icnc_median[t_mask]):.2f} L⁻¹")
print(f"  Peak mean diameter: {np.nanmax(diam_median[t_mask]):.1f} µm")
if rel_lwc_change is not None:
    print(f"  Min ΔLWC:           {np.nanmin(rel_lwc_change[t_mask]):.1f}%")
print(f"\nHOLIMO observations:")
for mission, data in hol_missions.items():
    if len(data['ice_conc']) > 0:
        print(f"  {mission}: ICNC={np.nanmedian(data['ice_conc']):.3f} cm⁻³, "
              f"ΔLWC={np.nanmedian(data['rel_lwc_change']):.1f}%")